# Duplication and Its Discontents — Colab Quickstart

Runs the full pipeline: generate data → fine-tune GPT-2 small at several duplication levels → evaluate memorization/generalization → evaluate diversity.

**Before running:** Runtime → Change runtime type → GPU.

Total runtime on a free Colab T4: roughly 30-60 minutes for all 4 duplication levels with GPT-2 small.

## 0. Setup

In [ ]:
!git clone https://github.com/YOUR_USERNAME/dup-project.git
%cd dup-project
!pip install -q -r requirements.txt

## 1. Generate the synthetic corpora (one per duplication level)

In [ ]:
%cd src
!python data_gen.py --out_dir ../data --dup_levels 1 5 20 100 --seed 0

## 2. Fine-tune GPT-2 small for each duplication level

This loops over every corpus and saves a checkpoint per duplication level. On a free T4 this is a few minutes per level for a small corpus.

In [ ]:
# epochs=1 is the clean design: with a single pass over the corpus, each fact occurrence
# is seen by the optimizer exactly once, so total training exposure to a fact string equals
# its duplication count exactly. (An earlier version of this notebook used --max_steps to
# equalize total compute across duplication levels -- that was a mistake: it gave the
# smallest corpus far more epochs than the largest, causing catastrophic overfitting. Stick
# with epochs=1 here.)
DUP_LEVELS = [1, 5, 20, 100]
for n in DUP_LEVELS:
    !python train.py --corpus ../data/corpus_dup{n}.txt --out_dir ../outputs/ckpt_dup{n} --epochs 1

## 3. Evaluate memorization vs. generalization

In [ ]:
!python evaluate_memorization.py \
    --ckpt_dir_pattern "../outputs/ckpt_dup{n}" \
    --dup_levels 1 5 20 100 \
    --facts_json ../data/facts.json \
    --out_dir ../outputs

In [ ]:
from IPython.display import Image
Image("../outputs/memorization_vs_generalization.png")

## 4. Evaluate generation diversity (the creativity angle)

In [ ]:
!python evaluate_diversity.py \
    --ckpt_dir_pattern "../outputs/ckpt_dup{n}" \
    --dup_levels 1 5 20 100 \
    --facts_json ../data/facts.json \
    --out_dir ../outputs \
    --n_samples 8

In [ ]:
Image("../outputs/diversity_vs_duplication.png")

## 5. Next steps

- Try more duplication levels (e.g. 2, 10, 50, 200) for a smoother curve.
- Try a second model size (e.g. `gpt2-medium`) to see if the effect scales.
- Write up the results: does memorization rise faster than generalization degrades? Does diversity collapse before or after memorization saturates?